In [9]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping
from functions import clean_data, split_data
from sklearn.model_selection import KFold


In [3]:
X, Y = clean_data("claims_train.csv")
X_train, X_val, y_train, y_val = split_data(X, Y, 0.2)
X_test, y_test = clean_data("claims_test.csv")

In [ ]:
def build_model(n_features, lr=0.001):
    model = keras.Sequential([
        keras.layers.Input(shape=(n_features,)),
        keras.layers.Dense(28, activation="relu", kernel_initializer="he_normal"),
        keras.layers.Dense(28, activation="relu", kernel_initializer="he_normal"),
        keras.layers.Dense(1, activation="softplus")
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=keras.losses.Poisson(),
        metrics=[keras.metrics.MeanSquaredError(name="mse")]
    )
    
    return model

def train_model_keras(X_train, y_train, X_val, y_val, learning_rate, batch_size, epochs):
    X_train = np.asarray(X_train)
    X_val   = np.asarray(X_val)
    y_train = np.asarray(y_train).reshape(-1, 1)
    y_val   = np.asarray(y_val).reshape(-1, 1)

    tf.keras.backend.clear_session()  
    model = build_model(n_features=X_train.shape[1], lr=learning_rate)

    es = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        callbacks=[es]
    )

    val_loss, val_mse = model.evaluate(X_val, y_val, verbose=0)
    return float(val_loss), float(val_mse)

def tune_batch_size_keras(X, y, batch_sizes, learning_rate=0.001, epochs=150, k=3, seed=42):
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)

    kf = KFold(n_splits=k, shuffle=True, random_state=seed)

    best_avg_loss = float("inf")
    best_batch_size = None

    for batch_size in batch_sizes:
        losses, mses = [], []

        for train_idx, val_idx in kf.split(X):
            X_tr, y_tr = X[train_idx], y[train_idx]
            X_va, y_va = X[val_idx], y[val_idx]

            val_loss, val_mse = train_model_keras(
                X_tr, y_tr,
                X_va, y_va,
                learning_rate=learning_rate,
                batch_size=batch_size,
                epochs=epochs
            )

            losses.append(val_loss)
            mses.append(val_mse)

        avg_loss = float(np.mean(losses))
        avg_mse  = float(np.mean(mses))

        print(f"batch={batch_size} -> avg Poisson: {avg_loss:.6f}, avg MSE: {avg_mse:.6f}")

        if avg_loss < best_avg_loss:
            best_avg_loss = avg_loss
            best_batch_size = batch_size

    print("Best batch size:", best_batch_size, "Best avg Poisson:", best_avg_loss)
    return best_batch_size, best_avg_loss


In [ ]:
batch_sizes = [16, 32, 64, 128, 256, 512]

best_batch_size, best_cv_poisson = tune_batch_size_keras(
    X_train, y_train,
    batch_sizes=batch_sizes,
    learning_rate=0.001,
    epochs=150,
    k=3
)
print(best_batch_size, best_cv_poisson)


batch=16 -> avg Poisson: 0.200251, avg MSE: 0.056013
batch=32 -> avg Poisson: 0.199822, avg MSE: 0.055905
batch=64 -> avg Poisson: 0.200071, avg MSE: 0.056028
batch=128 -> avg Poisson: 0.200282, avg MSE: 0.056041
batch=256 -> avg Poisson: 0.201046, avg MSE: 0.056108
batch=512 -> avg Poisson: 0.201174, avg MSE: 0.056152
Best batch size: 32 Best avg Poisson: 0.1998222271601359
32 0.1998222271601359


In [4]:
best_batch_size = 32

In [7]:
X_train_np = np.asarray(X_train)
y_train_np = np.asarray(y_train).reshape(-1, 1)

tf.keras.backend.clear_session()
final_model = build_model(n_features=X_train_np.shape[1], lr=0.001)

es = keras.callbacks.EarlyStopping(monitor="loss", patience=5, restore_best_weights=True)

final_model.fit(
    X_train_np, y_train_np,
    epochs=150,
    batch_size=best_batch_size,
    verbose=1,
    callbacks=[es]
)


Epoch 1/150

13561/13561 [==============================] - 11s 783us/step - loss: 0.2066 - mse: 0.0569
Epoch 2/150
13561/13561 [==============================] - 10s 732us/step - loss: 0.2031 - mse: 0.0564
Epoch 3/150
13561/13561 [==============================] - 10s 736us/step - loss: 0.2019 - mse: 0.0562
Epoch 4/150
13561/13561 [==============================] - 10s 751us/step - loss: 0.2009 - mse: 0.0561
Epoch 5/150
13561/13561 [==============================] - 10s 742us/step - loss: 0.2002 - mse: 0.0560
Epoch 6/150
13561/13561 [==============================] - 10s 750us/step - loss: 0.1998 - mse: 0.0559
Epoch 7/150
13561/13561 [==============================] - 10s 755us/step - loss: 0.1995 - mse: 0.0559
Epoch 8/150
13561/13561 [==============================] - 10s 747us/step - loss: 0.1993 - mse: 0.0558
Epoch 9/150
13561/13561 [==============================] - 10s 736us/step - loss: 0.1989 - mse: 0.0558
Epoch 10/150
13561/13561 [==============================] - 10s 740us/st

In [8]:
_, train_mse = final_model.evaluate(X_train, y_train)
print("Train MSE:", train_mse)

_, validation_mse = final_model.evaluate(X_val, y_val)
print("Validation MSE:", validation_mse)

_, test_mse = final_model.evaluate(X_test, y_test)
print("Test MSE:", test_mse)

13561/13561 [==============================] - 9s 643us/step - loss: 0.1959 - mse: 0.0555
Train MSE: 0.05547374486923218
3391/3391 [==============================] - 2s 632us/step - loss: 0.1962 - mse: 0.0551
Validation MSE: 0.05514516681432724
4238/4238 [==============================] - 3s 675us/step - loss: 0.2007 - mse: 0.0591
Test MSE: 0.05914340168237686
